# 🌾 KrishiSahayak - Plant Disease Detection Model Training

### Training Setup for Google Colab with GPU

**Expected Training Time:** 15-30 minutes with GPU

**Steps:**
1. Enable GPU: Runtime → Change runtime type → GPU (T4)
2. Run all cells in order
3. Download the trained model file at the end

In [ ]:
# Install required packages
!pip install -q kagglehub torch torchvision

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import kagglehub
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Download dataset from Kaggle
print("📥 Downloading PlantVillage dataset...")
path = kagglehub.dataset_download("emmarex/plantdisease")
print(f"✅ Dataset downloaded to: {path}")

# Check directory structure and find the correct path
print("\n🔍 Checking directory structure...")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 2:
            for dir_name in dirs[:5]:
                print(f"{indent}  {dir_name}/")

# Try to find the correct path with plant disease classes
possible_paths = [
    os.path.join(path, 'PlantVillage', 'PlantVillage'),
    os.path.join(path, 'PlantVillage'),
    path
]

DATASET_PATH = None
for test_path in possible_paths:
    if os.path.exists(test_path):
        # Check if this directory has subdirectories (classes)
        contents = os.listdir(test_path)
        subdirs = [d for d in contents if os.path.isdir(os.path.join(test_path, d))]
        # Look for typical plant disease folder names
        if any('Tomato' in d or 'Potato' in d or 'Pepper' in d for d in subdirs):
            DATASET_PATH = test_path
            print(f"\n✅ Found dataset at: {DATASET_PATH}")
            print(f"📋 Sample classes found: {subdirs[:3]}")
            break

if DATASET_PATH is None:
    raise FileNotFoundError(f"Could not find plant disease classes in {path}. Please check the dataset structure.")

In [ ]:
# Configuration
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
TRAIN_SPLIT = 0.8  # 80% train, 20% validation
NUM_CLASSES = None  # Will be determined from dataset
DISEASE_CLASSES = []

# Data augmentation and normalization
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("✅ Configuration set")

In [ ]:
# Load and split dataset
print("📂 Loading dataset...")

# Load entire dataset first
full_dataset = datasets.ImageFolder(DATASET_PATH, transform=None)

# Get classes from dataset
DISEASE_CLASSES = full_dataset.classes
NUM_CLASSES = len(DISEASE_CLASSES)
print(f"✅ Found {NUM_CLASSES} disease classes")
print(f"📋 Classes: {', '.join(DISEASE_CLASSES[:5])}... (and {NUM_CLASSES-5} more)")

# Split dataset into train and validation
train_size = int(TRAIN_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size

# Create datasets with transforms
train_data = datasets.ImageFolder(DATASET_PATH, transform=train_transform)
val_data = datasets.ImageFolder(DATASET_PATH, transform=val_transform)

# Apply the split with same random seed
train_data, _ = random_split(
    train_data, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
_, val_data = random_split(
    val_data, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Create data loaders
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ Training samples: {train_size}")
print(f"✅ Validation samples: {val_size}")
print(f"✅ Train/Val split: {int(TRAIN_SPLIT*100)}/{int((1-TRAIN_SPLIT)*100)}")

In [ ]:
# Load pre-trained ResNet-50 model
print("🔧 Loading ResNet-50 model...")
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze early layers (transfer learning)
for param in model.parameters():
    param.requires_grad = False

# Replace final layer for our classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, NUM_CLASSES)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

print("✅ Model ready for training")

In [ ]:
# Training loop
print("\n" + "="*60)
print("🚀 Starting training...")
print("="*60)

best_acc = 0.0
training_history = []

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")
    
    # Training phase
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if (i + 1) % 50 == 0:
            print(f"Batch [{i+1}/{len(train_loader)}] "
                  f"Loss: {running_loss/(i+1):.4f} "
                  f"Acc: {100.*correct/total:.2f}%")
    
    train_acc = 100. * correct / total
    train_loss = running_loss / len(train_loader)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100. * val_correct / val_total
    val_loss = val_loss / len(val_loader)
    
    print(f"\n📊 Epoch {epoch+1} Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss: {val_loss:.4f}   | Val Acc: {val_acc:.2f}%")
    
    # Save history
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc
    })
    
    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'accuracy': best_acc,
            'classes': DISEASE_CLASSES
        }, 'disease_model_best.pth')
        print(f"   ✅ New best model saved! (Accuracy: {best_acc:.2f}%)")

print("\n" + "="*60)
print(f"🎉 Training Complete!")
print(f"✅ Best Validation Accuracy: {best_acc:.2f}%")
print(f"✅ Model saved as: disease_model_best.pth")
print("="*60)

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in training_history]
train_losses = [h['train_loss'] for h in training_history]
val_losses = [h['val_loss'] for h in training_history]
train_accs = [h['train_acc'] for h in training_history]
val_accs = [h['val_acc'] for h in training_history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(epochs, train_losses, 'b-', label='Train Loss')
ax1.plot(epochs, val_losses, 'r-', label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(epochs, train_accs, 'b-', label='Train Accuracy')
ax2.plot(epochs, val_accs, 'r-', label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"\n📊 Final Results:")
print(f"Best Validation Accuracy: {best_acc:.2f}%")

In [ ]:
# Download the trained model
from google.colab import files

print("📥 Downloading trained model...")
files.download('disease_model_best.pth')
print("✅ Model downloaded! Upload this file to your backend/disease-ai/ folder")

## 🎯 Next Steps

1. Download the `disease_model_best.pth` file
2. Upload it to your `backend/disease-ai/` folder
3. The model is ready to use with your KrishiSahayak backend!

### Model Details:
- Architecture: ResNet-50 (pre-trained on ImageNet)
- Classes: 15 plant disease categories
- Input size: 224x224 RGB images
- Expected accuracy: 95%+ on validation set